In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end


In [ ]:
using Statistics, Plots

In [ ]:
grid =  bslLD.Grid([0.0,-7.0],[4*pi,7.0],[128,128],1, 0.0, 1)
simTime = bslLD.SimulationTime(0.01, 100.0; nmax=10000)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi) + 0.4*exp(-(v-4)^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.0001,initFuncv=initFuncv);
e = bslLD.empty_vectorfield(grid);


In [ ]:
mutable struct Diag
    rho::Vector
    f::Vector
    Ex::Vector
end
Diag() = Diag([], [], [])

function diags!(diags, f, rho, Ex, grid, simTime)
    simTime.step % 10 == 0 || return
    push!(diags.f, copy(f.data .- mean(f.data, dims=1)))
    push!(diags.rho, copy(rho.data[:]))
    push!(diags.Ex, copy(Ex.data[:]))
end

function step!(f, grid, simTime, diags)
    bslLD.advectX!(f, grid, simTime)
    rho = bslLD.compute_density(f, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.PoissonFieldSolver())
    bslLD.advectV!(f, grid, simTime, sol.E)
    diags!(diags, f, rho, sol.E[1], grid, simTime)
end


In [ ]:
diags = Diag()
while bslLD.continue_advection(simTime, true)
    step!(f, grid, simTime, diags)
    bslLD.advance!(simTime)
end


In [ ]:
num_frames = size(diags.f, 1)
frames_to_plot = 1:num_frames

animation = @animate for i in frames_to_plot
    heatmap(transpose(Array(diags.f[i])),
        title = "Frame $i",
        xlabel = "x",
        ylabel = "v",
    )

end
gif(animation, "fdiag_heatmap_animation.gif", fps = 10)

In [ ]:
plot(map(x->(mean((x.-mean(x)).^2)), Array.(diags.rho)), yscale=:log10)

In [ ]:
using FFTW

In [ ]:
plot(log.(abs.(fft(diags.Ex[end]))))